# Week 09 — Advanced Method Experiments

**Companion to:** A Spatial Data Warehouse  
**Course topic:** Data Warehousing

This notebook includes **four executable micro-experiments** that deepen understanding of the weekly methods. Each example uses a minimal, inspectable computation. Every experiment asks you to change a parameter, observe the result, and explain the difference.

These are **teaching demonstrations** using simulated or toy data; they are not measured observations about an actual city.

## Data setup and libraries
The same `DATA_ONLY.zip` may be uploaded into a new Colab runtime. On local Jupyter, keep the data folder at the project root.

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local Jupyter: data/ or ../data. Colab: upload DATA_ONLY.zip once per runtime.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA = next((p for p in candidates if (p / 'cells.csv').is_file()), None)
if DATA is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Cannot find data/. Extract the course ZIP and keep the data folder with the notebooks.') from exc
    print('Select DATA_ONLY.zip from the downloaded course package:')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
    if zip_name is None:
        raise FileNotFoundError('Please upload DATA_ONLY.zip and run this cell again.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(Path.cwd())
    DATA = Path('data')
assert (DATA / 'metadata.json').is_file(), 'Data package incomplete.'
print('Dataset folder:', DATA.resolve())
print('Synthetic teaching data: not field observations.')


## Experiment 1: Create a small SQLite schema

**Purpose.** Relational schemas separate descriptive cell dimensions from repeated observations.

**Scientific task:** Read the snippet, predict the output, run it, then explain the result and the assumptions.

In [ ]:
import sqlite3
con=sqlite3.connect(':memory:')
con.execute('CREATE TABLE dim_cell(cell_id TEXT PRIMARY KEY, sector TEXT)')
con.execute('INSERT INTO dim_cell VALUES (?,?)',('C001','NW'))
print(con.execute('SELECT * FROM dim_cell').fetchall())

### How to interpret this experiment

Relational schemas separate descriptive cell dimensions from repeated observations.

- `sqlite3` — Creates and queries a relational database from Python.
- `SELECT ` — Selects columns or computed aggregates from SQL tables.

**Questions:** (1) What does the printed output measure? (2) What happens if one input or parameter changes? (3) How could this method be extended to an independent geographic dataset?

## Experiment 2: COUNT null versus rows

**Purpose.** SQL COUNT(column) ignores NULL while COUNT(*) includes every row.

**Scientific task:** Read the snippet, predict the output, run it, then explain the result and the assumptions.

In [ ]:
import sqlite3
con=sqlite3.connect(':memory:')
con.execute('CREATE TABLE obs(x REAL)')
con.executemany('INSERT INTO obs VALUES (?)',[(30.,),(None,),(33.,)])
print(con.execute('SELECT COUNT(*),COUNT(x),AVG(x) FROM obs').fetchone())

### How to interpret this experiment

SQL COUNT(column) ignores NULL while COUNT(*) includes every row.

- `sqlite3` — Creates and queries a relational database from Python.
- `SELECT ` — Selects columns or computed aggregates from SQL tables.

**Questions:** (1) What does the printed output measure? (2) What happens if one input or parameter changes? (3) How could this method be extended to an independent geographic dataset?

## Experiment 3: SQL group-by from pandas

**Purpose.** SQL GROUP BY requires clearly defined spatial and temporal grouping keys.

**Scientific task:** Read the snippet, predict the output, run it, then explain the result and the assumptions.

In [ ]:
import pandas as pd, sqlite3
obs=pd.read_csv(DATA/'observations_clean.csv')
con=sqlite3.connect(':memory:')
obs.to_sql('observations',con,index=False)
print(pd.read_sql_query('SELECT date,COUNT(lst_c) AS n FROM observations GROUP BY date',con))

### How to interpret this experiment

SQL GROUP BY requires clearly defined spatial and temporal grouping keys.

- `sqlite3` — Creates and queries a relational database from Python.
- `pd.read_csv` — Reads CSV text into a pandas DataFrame. Inspect head() and column types after loading.
- `SELECT ` — Selects columns or computed aggregates from SQL tables.

**Questions:** (1) What does the printed output measure? (2) What happens if one input or parameter changes? (3) How could this method be extended to an independent geographic dataset?

## Experiment 4: Check composite keys

**Purpose.** A composite key guards against duplicate cell-date facts.

**Scientific task:** Read the snippet, predict the output, run it, then explain the result and the assumptions.

In [ ]:
import pandas as pd
obs=pd.read_csv(DATA/'observations_clean.csv')
keys=['cell_id','date']
assert not obs.duplicated(keys).any()
print('Unique fact grain verified')

### How to interpret this experiment

A composite key guards against duplicate cell-date facts.

- `pd.read_csv` — Reads CSV text into a pandas DataFrame. Inspect head() and column types after loading.
- `assert` — Checks a claimed condition and stops if it is false.

**Questions:** (1) What does the printed output measure? (2) What happens if one input or parameter changes? (3) How could this method be extended to an independent geographic dataset?

## Reflection and extension

Compare the four experiments. Which operation depends on spatial location? Which depends on attribute values? Which requires an independent evaluation design? Record parameter values and assumptions.

**Suggested submission:** One figure or table, annotated Python code, and one paragraph comparing the changed setting to the baseline.